# Personne 3 — Calcul de l'indicateur d'accessibilité aux médecins généralistes

Ce notebook combine :

- le voisinage routier calculé par la **personne 1** (couples de communes à moins de 20 min et poids associés) ;
- l'offre et la demande communales calculées par la **personne 2**.

L'indicateur est inspiré de l'**APL / E2SFCA** de la DREES.

**Attention :** il ne s'agit pas d'une reproduction exacte de l'APL officiel, car notre offre est mesurée par un nombre de médecins généralistes et non par leur activité réelle.

## 1. Principe

Pour chaque commune d'offre $j$, on calcule d'abord :

$$R_j = \frac{O_j}{\sum_i D_i \, w_{ij}}$$

avec :

- $O_j$ : nombre de médecins généralistes dans la commune $j$ ;
- $D_i$ : demande de la commune $i$ (population totale **ou** population standardisée selon l'âge) ;
- $w_{ij}$ : poids associé au temps routier entre $i$ et $j$.

Puis, pour chaque commune de résidence $i$ :

$$A_i = \sum_j R_j \, w_{ij}$$

On multiplie ensuite par 100 000 pour obtenir un nombre de médecins généralistes accessibles **pour 100 000 habitants**.

On calcule deux versions :

| Indicateur | Demande $D_i$ |
|---|---|
| `accessibilite_brute` | `population` |
| `accessibilite_ajustee_age` | `demande_standardisee` |

**Propriété de contrôle** : comme les mêmes poids sont utilisés aux deux étapes, on doit avoir
$\sum_i D_i A_i = \sum_j O_j$ : chaque médecin est « distribué » exactement une fois entre les habitants. On le vérifie en section 7.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## 2. Paramètres et chemins

- Entrées : sorties des personnes 1 et 2 (lecture en `https://` si le fichier est public, ou en `s3://`).
- Sortie : enregistrée en local puis, si `UPLOAD_MINIO = True`, envoyée sur notre bucket MinIO.

In [ ]:
# --- Entrées (sorties des personnes 1 et 2, dans le bucket de Mariama) -------
CHEMIN_VOISINAGE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "voisinage_routier_20min.parquet"
)
CHEMIN_OFFRE_DEMANDE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "offre_demande_mg_commune.parquet"
)

# --- Sorties ----------------------------------------------------------------
NOM_FICHIER_SORTIE = "indicateur.parquet"

# Dossier personnel (toujours accessible en écriture, et hors du repo GitHub)
DOSSIER_SORTIE = Path.home() / "donnees_projet"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE_LOCAL = DOSSIER_SORTIE / NOM_FICHIER_SORTIE

UPLOAD_MINIO = True   # envoie le résultat sur ton stockage SSPCloud
CHEMIN_S3_SORTIE = f"emma457/Projet_python/{NOM_FICHIER_SORTIE}"

# --- Paramètres de l'indicateur ---------------------------------------------
ECHELLE = 100_000   # résultat exprimé pour 100 000 habitants

In [ ]:
def systeme_fichiers_s3():
    # Connexion à MinIO avec les identifiants du service SSPCloud
    import s3fs
    return s3fs.S3FileSystem(
        client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
    )


def lire_parquet(chemin):
    # Lit un parquet en local, en https (fichier public) ou en s3:// (MinIO)
    chemin = str(chemin)
    if chemin.startswith("s3://"):
        with systeme_fichiers_s3().open(chemin, "rb") as f:
            return pd.read_parquet(f)
    return pd.read_parquet(chemin)

## 3. Chargement des données

In [ ]:
voisinage = lire_parquet(CHEMIN_VOISINAGE)
offre_demande = lire_parquet(CHEMIN_OFFRE_DEMANDE)

print("Voisinage :", voisinage.shape)
print("Offre-demande :", offre_demande.shape)

display(voisinage.head())
display(offre_demande.head())

## 4. Contrôles et harmonisation des entrées

Avant de calculer, vérifions que les deux tables « se parlent » :

1. colonnes attendues présentes ;
2. codes communes au même format (texte sur 5 caractères) ;
3. pas de doublons ;
4. pas de valeurs manquantes ou négatives, poids compris entre 0 et 1 ;
5. aucun médecin ni habitant « perdu » parce que sa commune n'est pas dans le voisinage ;
6. chaque commune est voisine d'elle-même (temps 0, poids 1).

In [ ]:
# 4.1 Colonnes nécessaires
colonnes_voisinage = {"CODGEO_origine", "CODGEO_destination", "poids_distance"}
colonnes_offre_demande = {
    "CODGEO", "LIBGEO", "population",
    "demande_standardisee", "nb_medecins_generalistes",
}

manquantes_voisinage = colonnes_voisinage - set(voisinage.columns)
manquantes_offre = colonnes_offre_demande - set(offre_demande.columns)

if manquantes_voisinage:
    raise ValueError(f"Colonnes manquantes dans le voisinage : {manquantes_voisinage}")
if manquantes_offre:
    raise ValueError(f"Colonnes manquantes dans offre-demande : {manquantes_offre}")

print("Colonnes nécessaires présentes.")

In [ ]:
# 4.2 Codes communes : texte sur 5 caractères ('1001' -> '01001', '1001.0' -> '01001', Corse 2A/2B conservée)
def normaliser_codgeo(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

for col in ["CODGEO_origine", "CODGEO_destination"]:
    voisinage[col] = normaliser_codgeo(voisinage[col])
offre_demande["CODGEO"] = normaliser_codgeo(offre_demande["CODGEO"])

print("Exemples de codes :", offre_demande["CODGEO"].sample(5, random_state=0).tolist())

In [ ]:
# 4.3 Doublons
if offre_demande["CODGEO"].duplicated().any():
    raise ValueError("La table offre-demande contient des CODGEO dupliqués.")

n_doublons = voisinage.duplicated(["CODGEO_origine", "CODGEO_destination"]).sum()
print("Couples origine-destination en double :", n_doublons)
if n_doublons:
    voisinage = voisinage.drop_duplicates(["CODGEO_origine", "CODGEO_destination"])

print("Nombre de communes :", offre_demande["CODGEO"].nunique())
print("Nombre de couples origine-destination :", len(voisinage))

In [ ]:
# 4.4 Valeurs manquantes, négatives, poids
cols_num = ["population", "demande_standardisee", "nb_medecins_generalistes"]

print("Valeurs manquantes :")
print(offre_demande[cols_num].isna().sum())
offre_demande[cols_num] = offre_demande[cols_num].fillna(0)

if (offre_demande[cols_num] < 0).any().any():
    raise ValueError("Valeurs négatives dans offre-demande.")

print(
    f"\nPoids : min = {voisinage['poids_distance'].min():.3f}, "
    f"max = {voisinage['poids_distance'].max():.3f}"
)
print("Répartition des poids :")
print(voisinage["poids_distance"].round(3).value_counts().sort_index())

if not voisinage["poids_distance"].between(0, 1).all():
    raise ValueError("Des poids sortent de [0, 1].")

# Un poids nul n'apporte rien au calcul
voisinage = voisinage[voisinage["poids_distance"] > 0].copy()

In [ ]:
# 4.5 Communes qui ne trouvent pas leur place dans le voisinage
offre_perdue = offre_demande[
    (offre_demande["nb_medecins_generalistes"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_destination"])
]
population_perdue = offre_demande[
    (offre_demande["population"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_origine"])
]
codes_voisinage_inconnus = (
    set(voisinage["CODGEO_origine"]) | set(voisinage["CODGEO_destination"])
) - set(offre_demande["CODGEO"])

print(
    f"Communes avec médecins absentes des destinations : {len(offre_perdue)} "
    f"({offre_perdue['nb_medecins_generalistes'].sum():.0f} médecins)"
)
print(
    f"Communes peuplées absentes des origines : {len(population_perdue)} "
    f"({population_perdue['population'].sum():,.0f} habitants)"
)
print(f"Codes du voisinage absents d'offre-demande : {len(codes_voisinage_inconnus)}")

# À regarder en priorité : arrondissements Paris / Lyon / Marseille, communes nouvelles
display(offre_perdue.sort_values("nb_medecins_generalistes", ascending=False).head(10))

In [ ]:
# 4.6 Chaque commune doit être voisine d'elle-même (temps 0 -> poids 1)
deja_presentes = set(
    voisinage.loc[
        voisinage["CODGEO_origine"] == voisinage["CODGEO_destination"],
        "CODGEO_origine",
    ]
)
a_ajouter = sorted(set(offre_demande["CODGEO"]) - deja_presentes)
print("Couples (i, i) ajoutés :", len(a_ajouter))

if a_ajouter:
    voisinage = pd.concat(
        [
            voisinage,
            pd.DataFrame({
                "CODGEO_origine": a_ajouter,
                "CODGEO_destination": a_ajouter,
                "poids_distance": 1.0,
            }),
        ],
        ignore_index=True,
    )

In [ ]:
# 4.7 Comparaison des deux demandes
total_pop = offre_demande["population"].sum()
total_std = offre_demande["demande_standardisee"].sum()
print(f"Population totale         : {total_pop:,.0f}")
print(f"Demande standardisée      : {total_std:,.0f}")
print(f"Rapport                   : {total_std / total_pop:.3f}")

## 5. Fonction de calcul (E2SFCA)

Les deux étapes sont regroupées dans une fonction, pour pouvoir :

- calculer la version brute et la version ajustée avec le même code ;
- la réutiliser dans le notebook 04 avec d'autres poids (sensibilité).

In [ ]:
def calcul_accessibilite(voisinage, offre_demande, col_demande,
                         col_offre="nb_medecins_generalistes",
                         col_poids="poids_distance", echelle=ECHELLE):
    # Renvoie :
    #   accessibilite : Series indexée par CODGEO (médecins accessibles pour `echelle` habitants)
    #   ratio         : Series indexée par CODGEO des communes d'offre (R_j)
    demande = offre_demande.set_index("CODGEO")[col_demande]
    offre = offre_demande.set_index("CODGEO")[col_offre]

    couples = voisinage[["CODGEO_origine", "CODGEO_destination", col_poids]].copy()

    # Étape 1 : demande pondérée qui pèse sur chaque commune d'offre j
    couples["demande_ponderee"] = (
        couples["CODGEO_origine"].map(demande).fillna(0) * couples[col_poids]
    )
    demande_accessible = couples.groupby("CODGEO_destination")["demande_ponderee"].sum()

    # Ratio offre / demande R_j
    offre_j = offre.reindex(demande_accessible.index).fillna(0)
    ratio = (offre_j / demande_accessible).where(demande_accessible > 0, 0.0)
    ratio = ratio[offre_j > 0]

    sans_demande = offre_j[(offre_j > 0) & (demande_accessible == 0)]
    if len(sans_demande):
        print(f"  Attention : {len(sans_demande)} communes ont des médecins mais "
              f"aucune demande accessible ({sans_demande.sum():.0f} médecins ignorés).")

    # Étape 2 : somme des R_j accessibles depuis chaque commune i
    couples["contribution"] = (
        couples["CODGEO_destination"].map(ratio).fillna(0) * couples[col_poids]
    )
    accessibilite = couples.groupby("CODGEO_origine")["contribution"].sum()

    accessibilite = accessibilite.reindex(offre_demande["CODGEO"]).fillna(0) * echelle
    return accessibilite, ratio

## 6. Calcul des deux versions

In [ ]:
print("Version brute (population totale)")
acc_brute, ratio_brute = calcul_accessibilite(voisinage, offre_demande, "population")

print("Version ajustée par l'âge (demande standardisée)")
acc_age, ratio_age = calcul_accessibilite(voisinage, offre_demande, "demande_standardisee")

resultat = offre_demande[
    ["CODGEO", "LIBGEO", "population", "demande_standardisee", "nb_medecins_generalistes"]
].copy()

resultat["accessibilite_brute"] = resultat["CODGEO"].map(acc_brute).values
resultat["accessibilite_ajustee_age"] = resultat["CODGEO"].map(acc_age).values

# Densité « naïve » : médecins de la commune / habitants de la commune (pour comparaison)
resultat["densite_mg"] = np.where(
    resultat["population"] > 0,
    resultat["nb_medecins_generalistes"] / resultat["population"] * ECHELLE,
    np.nan,
)

# Effet de l'âge : < 1 = la commune perd en accessibilité quand on tient compte de l'âge
resultat["ratio_age_brute"] = np.where(
    resultat["accessibilite_brute"] > 0,
    resultat["accessibilite_ajustee_age"] / resultat["accessibilite_brute"],
    np.nan,
)

print("\nTable finale :", resultat.shape)
resultat.head()

## 7. Contrôles du résultat

1. **Conservation** : $\sum_i D_i A_i = \sum_j O_j$ pour chaque version.
2. Pas de doublons, pas de valeurs négatives ou infinies.
3. Ordre de grandeur : la moyenne pondérée par la population doit être proche de la densité nationale.

In [ ]:
def verifier_conservation(accessibilite, ratio, col_demande, nom):
    od = offre_demande.set_index("CODGEO")
    offre_distribuee = od["nb_medecins_generalistes"].reindex(ratio.index).sum()
    reconstruit = (od[col_demande] * accessibilite.reindex(od.index) / ECHELLE).sum()
    ecart = abs(reconstruit - offre_distribuee) / offre_distribuee
    print(f"{nom:<8} médecins distribués = {offre_distribuee:,.1f} | "
          f"somme D_i x A_i = {reconstruit:,.1f} | écart relatif = {ecart:.1e}")
    if ecart > 1e-6:
        raise ValueError("Conservation non respectée : problème dans le calcul.")

verifier_conservation(acc_brute, ratio_brute, "population", "brute")
verifier_conservation(acc_age, ratio_age, "demande_standardisee", "âge")

print("\nMédecins au total :", offre_demande["nb_medecins_generalistes"].sum())

In [ ]:
cols_acc = ["accessibilite_brute", "accessibilite_ajustee_age"]

print("Doublons :", resultat["CODGEO"].duplicated().sum())
print("Valeurs négatives :", (resultat[cols_acc] < 0).sum().to_dict())
print("Valeurs infinies ou manquantes :", (~np.isfinite(resultat[cols_acc])).sum().to_dict())

peuplees = resultat[resultat["population"] > 0]
densite_nationale = (
    resultat["nb_medecins_generalistes"].sum() / resultat["population"].sum() * ECHELLE
)
moyenne_ponderee = np.average(peuplees["accessibilite_brute"], weights=peuplees["population"])
print(f"\nDensité nationale : {densite_nationale:.1f} MG pour 100 000 hab.")
print(f"Accessibilité brute moyenne (pondérée par la population) : {moyenne_ponderee:.1f}")